# IE 7615 – Deep Learning for AI
## Milestone 1: Object Recognition — 2 Classification Models + 2 Object Detection Models

This notebook runs the complete Milestone 1 pipeline on the final merged dataset (73 Object IDs):

| # | Model | Task | Approach |
|---|---|---|---|
| 1 | **Custom CNN** | Classification | Trained from scratch (baseline) |
| 2 | **EfficientNet-B0** | Classification | ImageNet-pretrained, fine-tuned (transfer learning) |
| 3 | **YOLO11s** | Object detection | One-stage detector, COCO-pretrained, fine-tuned |
| 4 | **Faster R-CNN (ResNet50-FPN v2)** | Object detection | Two-stage detector, COCO-pretrained, fine-tuned |

**Sections**
- **Part A – Preprocessing:** validation → exact-duplicate removal → resize → RGB → stratified split (shared by all four models)
- **Part B – Classification:** Custom CNN and EfficientNet-B0
- **Part C – Detection:** automatic bounding-box generation, YOLO11s, Faster R-CNN
- **Part D – Final comparison** of all four models on the same test set

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`.
2. If the dataset folder is under *Shared with me*, right-click it → *Organize → Add shortcut* → My Drive.
3. `Runtime → Run all` (≈ 2–2.5 h on a T4). Keep the tab open.

**If Colab disconnects:** just *Run all* again. Every finished step (preprocessing, trained models, generated boxes) is saved to Drive and is reloaded instead of being redone.

## Setup

In [ ]:
!pip -q install ultralytics torchmetrics pycocotools

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import sys, io, re, json, time, random, shutil, hashlib, warnings, contextlib
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, top_k_accuracy_score)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision import transforms, models

Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Mount Google Drive in Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

### Paths and settings — change only `DATASET_ROOT` / `OUTPUT_ROOT` if needed

In [ ]:
# Final merged dataset: one folder per Object ID (images_OBJ001, images_OBJ002, ...)
DATASET_ROOT = Path("/content/drive/MyDrive/IE7615 Deep Learning for AI")

# All outputs (results, figures, trained models) are saved here
OUTPUT_ROOT = Path("/content/drive/MyDrive/IE_7615/Milestone_1_Results/Four_Model_Comparison")

# Fast local working copies (Colab's local disk is much faster than Drive)
WORK_DIR = Path("/content") if IN_COLAB else Path.home() / "milestone1_work"
DATA_DIR = WORK_DIR / "prep_224"        # preprocessed 224x224 images
YOLO_DIR = WORK_DIR / "yolo_ds"         # dataset in YOLO format

PREP_OUT, CLS_DIR, DET_DIR = OUTPUT_ROOT / "preprocessing", OUTPUT_ROOT / "classification", OUTPUT_ROOT / "detection"
for d in (OUTPUT_ROOT, PREP_OUT, CLS_DIR, DET_DIR, WORK_DIR): d.mkdir(parents=True, exist_ok=True)

SEED        = 42
IMG_SIZE    = 224
VAL_FRAC    = 0.15
TEST_FRAC   = 0.15
RESIZE_MODE = "pad"    # non-square images: letterbox (keeps the whole object)
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_NUMBER_OF_CLASSES = 73

def find_dataset_root(start, max_depth=3):
    # Find the folder containing the images_OBJ### subfolders (handles shortcuts with a different name)
    if not start.exists(): return None
    for root, dirs, _ in os.walk(start):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if sum(bool(re.search(r"OBJ\d+$", d, re.I)) for d in dirs) >= 10: return Path(root)
        if len(Path(root).relative_to(start).parts) >= max_depth: dirs[:] = []
    return None

if not DATASET_ROOT.exists():
    print("Dataset not at", DATASET_ROOT, "- searching Drive for the images_OBJ### folders ...")
    for start in [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]:
        found = find_dataset_root(start)
        if found: DATASET_ROOT = found; break

print("Dataset path:", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())
print("Output path: ", OUTPUT_ROOT)
if not DATASET_ROOT.exists():
    mydrive = Path("/content/drive/MyDrive")
    print("\nFolders in your My Drive:", sorted(p.name for p in mydrive.iterdir() if p.is_dir()) if mydrive.exists() else "Drive not mounted")
assert DATASET_ROOT.exists(), ("Dataset not found. Open the dataset link, click the folder name > Organize > Add shortcut > My Drive, "
                               "then rerun. Or set DATASET_ROOT to the folder that contains images_OBJ001, images_OBJ002, ...")

### Final Object IDs and object names (from the updated *Milestone-1 OBJECTS – LIST OF OBJECTS* sheet)

In [ ]:
OBJECT_NAMES = {
    'OBJ001': 'Basketball', 'OBJ002': 'water bottle', 'OBJ003': 'NoteBook', 'OBJ004': 'Toy Car',
    'OBJ005': 'Computer mouse', 'OBJ006': 'Capybara Toy', 'OBJ007': 'Apple', 'OBJ008': 'Cat',
    'OBJ009': 'Marshall Speaker', 'OBJ010': 'Nike sneakers', 'OBJ011': 'mug', 'OBJ012': 'Computer Keyboard',
    'OBJ013': 'Mobile phone', 'OBJ014': 'Sunglasses', 'OBJ015': 'Water Tumbler', 'OBJ016': 'Sandisk Portable SSD',
    'OBJ017': 'Controller', 'OBJ018': 'Rubiks cube', 'OBJ019': 'Plant', 'OBJ020': 'Airpod',
    'OBJ021': 'Toy Bunny', 'OBJ022': 'Guitar', 'OBJ023': 'Bracelet', 'OBJ024': 'Plush doll',
    'OBJ025': 'Watch', 'OBJ026': 'Red Sox Cap', 'OBJ027': 'Tennis ball', 'OBJ028': 'banana',
    'OBJ029': 'Orange', 'OBJ030': 'Tennis racket', 'OBJ031': 'UNO card box', 'OBJ032': 'Sound Card',
    'OBJ033': 'Duck', 'OBJ034': 'Table Tennis Racket', 'OBJ035': 'Badminton racket', 'OBJ036': 'Pencil case',
    'OBJ037': 'Candle', 'OBJ038': 'Perfume bottle', 'OBJ039': 'Scissors', 'OBJ040': 'Clutcher',
    'OBJ041': 'Table fan', 'OBJ042': 'Mouth guard case', 'OBJ043': 'Swimming Goggels', 'OBJ044': 'Anime Figurine',
    'OBJ045': 'Wayfarer Sunglasses', 'OBJ046': 'iWatch', 'OBJ047': 'Backpack', 'OBJ048': 'Wireless powerbank',
    'OBJ049': 'Pickleball bat', 'OBJ050': 'Hairbrush', 'OBJ051': 'tissue', 'OBJ052': 'Spoon',
    'OBJ053': 'Moisturizer', 'OBJ054': 'Purse', 'OBJ055': 'Scientific Calculator', 'OBJ056': "Trader Joe's Bag",
    'OBJ057': 'Power Bank', 'OBJ058': 'Footstool', 'OBJ059': 'Comb', 'OBJ060': 'Sleep Mask',
    'OBJ061': 'Card wallet', 'OBJ062': 'Headphones', 'OBJ063': 'Measuring Bottle', 'OBJ064': 'RGB Projector',
    'OBJ065': 'Analog watch', 'OBJ066': 'hair oil', 'OBJ067': "Men's Wallet", 'OBJ068': 'Neck Pillow',
    'OBJ069': 'knife', 'OBJ070': 'Astronaut projector', 'OBJ071': 'Sloth Plushie', 'OBJ072': 'Dog',
    'OBJ073': 'Bowl',
}

def disp(obj_id, maxlen=22):
    name = OBJECT_NAMES.get(obj_id)
    return f"{obj_id} · {name[:maxlen]}" if name else obj_id
print(len(OBJECT_NAMES), "Object IDs in the official list")

In [ ]:
# Device, reproducibility and shared helpers
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

if torch.cuda.is_available():            DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():  DEVICE = torch.device("mps")
else:                                    DEVICE = torch.device("cpu")
def amp():          # mixed precision on NVIDIA GPUs
    return torch.autocast("cuda") if DEVICE.type == "cuda" else contextlib.nullcontext()
def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    elif DEVICE.type == "mps": torch.mps.synchronize()
def free_mem():
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    elif DEVICE.type == "mps": torch.mps.empty_cache()
NUM_WORKERS = 2 if (IN_COLAB or sys.platform.startswith("linux")) else 0
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type == "cpu": print("WARNING: no GPU - set Runtime > Change runtime type > T4 GPU, otherwise training will take many hours.")

---
# Part A — Preprocessing (shared by all four models)

Workflow recommended by the TAs: **raw dataset → image validation → exact-duplicate check → resize/downsample → RGB conversion → stratified train/val/test split.** Augmentation and normalization are applied later, per model.

## A1. Inventory and Object ID verification

In [ ]:
OBJ_RE = re.compile(r"(?:^|[^A-Za-z])OBJ\s*0*(\d+)$", re.IGNORECASE)   # "images_OBJ001" -> OBJ001

def infer_label(p: Path):
    for part in p.relative_to(DATASET_ROOT).parts[:-1]:
        m = OBJ_RE.search(part)
        if m: return f"OBJ{int(m.group(1)):03d}"
    return None

all_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()
             and not any(part.startswith(".") for part in p.relative_to(DATASET_ROOT).parts)]
files = [p for p in all_files if infer_label(p)]
ignored = [p for p in all_files if not infer_label(p)]
if ignored: print(f"Ignored {len(ignored)} files outside Object ID folders, e.g.", [p.name for p in ignored[:5]])

inv = pd.DataFrame({"path": [str(p) for p in files], "label": [infer_label(p) for p in files],
                    "ext": [p.suffix.lower() for p in files]}).sort_values(["label", "path"]).reset_index(drop=True)
non_img = inv[~inv.ext.isin(IMAGE_EXTS)]
folder_ids = sorted(inv.label.unique())

print(f"Files found: {len(inv)} | Object ID folders: {len(folder_ids)}")
print("Extensions:", dict(Counter(inv.ext)))
if len(non_img): print(f"Non-image files (excluded): {len(non_img)}")
print("In dataset but NOT in official list:", sorted(set(folder_ids) - set(OBJECT_NAMES)) or "none")
print("In official list but NOT in dataset:", sorted(set(OBJECT_NAMES) - set(folder_ids)) or "none")
assert len(folder_ids) == EXPECTED_NUMBER_OF_CLASSES, f"Expected {EXPECTED_NUMBER_OF_CLASSES} Object IDs, found {len(folder_ids)}"
print("Object IDs verified.")

## A2. Image validation and content hashing (cached to Drive)

In [ ]:
INV_CACHE = PREP_OUT / "raw_inventory.csv"

def inspect_file(path):
    rec = {"path": path, "ok": False, "md5": None, "width": None, "height": None, "mode": None, "error": None}
    try:
        data = Path(path).read_bytes()
        rec["md5"] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im: im.verify()
        with Image.open(io.BytesIO(data)) as im:
            w, h = im.size
            if im.getexif().get(0x0112, 1) in (5, 6, 7, 8): w, h = h, w
            rec.update(width=w, height=h, mode=im.mode, ok=True)
    except Exception as e:
        rec["error"] = repr(e)[:200]
    return rec

img_inv = inv[inv.ext.isin(IMAGE_EXTS)]
if INV_CACHE.exists() and set(pd.read_csv(INV_CACHE).path) == set(img_inv.path):
    meta_df = pd.read_csv(INV_CACHE); print("Loaded cached inventory.")
else:
    with ThreadPoolExecutor(16) as ex:
        meta_df = pd.DataFrame(list(tqdm(ex.map(inspect_file, img_inv.path), total=len(img_inv), desc="Validating")))
    meta_df = img_inv.merge(meta_df, on="path"); meta_df.to_csv(INV_CACHE, index=False)

okm = meta_df.ok.astype(bool); good = meta_df[okm]
print(f"Readable: {okm.sum()} | Unreadable/corrupt: {(~okm).sum()}")
if (~okm).any(): display(meta_df[~okm][["label", "path", "error"]])
print("Most common dimensions:", Counter(zip(good.width.astype(int), good.height.astype(int))).most_common(5))
not224 = good[(good.width != IMG_SIZE) | (good.height != IMG_SIZE)]
print(f"Images not {IMG_SIZE}x{IMG_SIZE}: {len(not224)} (resized, not dropped)")
print("Color modes:", dict(Counter(good["mode"])))

## A3. Exact-duplicate removal (before splitting, to prevent train/test leakage)

In [ ]:
ok = good.copy()
ok["dup_rank"] = ok.groupby(["label", "md5"]).cumcount()
lab_per_hash = ok.groupby("md5")["label"].nunique()
cross = set(lab_per_hash[lab_per_hash > 1].index)       # same image under 2+ Object IDs -> ambiguous label

excl  = [(r.path, r.label, "not an image file") for r in non_img.itertuples()]
excl += [(r.path, r.label, "unreadable/corrupt") for r in meta_df[~okm].itertuples()]
excl += [(r.path, r.label, "exact duplicate within class") for r in ok[ok.dup_rank > 0].itertuples()]
excl += [(r.path, r.label, "exact duplicate across classes (ambiguous label)") for r in ok[(ok.dup_rank == 0) & ok.md5.isin(cross)].itertuples()]
excluded_df = pd.DataFrame(excl, columns=["path", "label", "reason"])
excluded_df.to_csv(PREP_OUT / "excluded_images.csv", index=False)

clean = ok[(ok.dup_rank == 0) & ~ok.md5.isin(cross)].reset_index(drop=True)
print(f"Clean images: {len(clean)} / {len(inv)} files")
print(excluded_df.reason.value_counts().to_string() if len(excluded_df) else "nothing excluded")
counts = clean.label.value_counts().sort_index()
print(f"Images per Object ID: min={counts.min()}  median={int(counts.median())}  max={counts.max()}")

fig, ax = plt.subplots(figsize=(18, 4))
ax.bar(counts.index, counts.values); ax.set_title("Clean images per Object ID"); ax.set_ylabel("images")
ax.tick_params(axis="x", rotation=90, labelsize=7); plt.tight_layout()
plt.savefig(PREP_OUT / "class_distribution.png", dpi=150); plt.show()

## A4. Resize / downsample + RGB conversion (local copy; zipped to Drive for reuse)

In [ ]:
PREP_ZIP = PREP_OUT / "prep_224.zip"
clean["rel_path"] = clean.label + "/" + clean.md5 + ".jpg"
if not DATA_DIR.exists() and PREP_ZIP.exists():
    print("Restoring preprocessed images from Drive..."); shutil.unpack_archive(str(PREP_ZIP), str(DATA_DIR.parent))

def load_and_resize(path, size=IMG_SIZE, mode=RESIZE_MODE):
    im = Image.open(path)
    if im.format == "JPEG": im.draft("RGB", (size * 2, size * 2))      # fast decode for very large JPEGs
    im = ImageOps.exif_transpose(im)
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (255, 255, 255, 255)); bg.alpha_composite(im); im = bg
    im = im.convert("RGB")
    w, h = im.size
    if abs(w / h - 1) < 0.02: return im.resize((size, size), Image.Resampling.LANCZOS)
    if mode == "crop": return ImageOps.fit(im, (size, size), Image.Resampling.LANCZOS)
    return ImageOps.pad(im, (size, size), Image.Resampling.LANCZOS, color=(0, 0, 0))

def prep_one(row):
    out = DATA_DIR / row["rel_path"]
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        load_and_resize(row["path"]).save(out, quality=95)
    return True

missing = [r for r in clean.to_dict("records") if not (DATA_DIR / r["rel_path"]).exists()]
if missing:
    with ThreadPoolExecutor(16) as ex:
        list(tqdm(ex.map(prep_one, missing), total=len(missing), desc="Resizing"))
    shutil.make_archive(str(PREP_ZIP.with_suffix("")), "zip", root_dir=str(DATA_DIR.parent), base_dir=DATA_DIR.name)
print("Preprocessed images ready:", len(clean))
clean["img"] = [str(DATA_DIR / p) for p in clean.rel_path]

rng = np.random.default_rng(SEED)
show = rng.choice(folder_ids, size=min(24, len(folder_ids)), replace=False)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, lab in zip(axes.flat, show):
    ax.imshow(Image.open(clean[clean.label == lab].sample(1, random_state=SEED).img.iloc[0]))
    ax.set_title(disp(lab, 16), fontsize=8); ax.axis("off")
for ax in axes.flat[len(show):]: ax.axis("off")
plt.tight_layout(); plt.savefig(PREP_OUT / "sample_images.png", dpi=120); plt.show()

## A5. Stratified 70 / 15 / 15 split

In [ ]:
classes = sorted(clean.label.unique()); NUM_CLASSES = len(classes)
clean["y"] = clean.label.map({c: i for i, c in enumerate(classes)})
train_df, temp_df = train_test_split(clean, test_size=VAL_FRAC + TEST_FRAC, stratify=clean.y, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=temp_df.y, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
assert not (set(train_df.md5) & (set(val_df.md5) | set(test_df.md5))), "leakage!"
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert d.label.nunique() == NUM_CLASSES
    d[["rel_path", "label", "y", "md5", "path"]].to_csv(PREP_OUT / f"split_{name}.csv", index=False)
pd.DataFrame({"object_id": classes, "object_name": [OBJECT_NAMES.get(c, "") for c in classes],
              "clean_images": counts.reindex(classes).values}).to_csv(PREP_OUT / "object_ids.csv", index=False)

prep_summary = {"raw_files": int(len(inv)), "readable_images": int(okm.sum()), "clean_images": int(len(clean)),
                "excluded_by_reason": excluded_df.reason.value_counts().to_dict(), "images_not_224": int(len(not224)),
                "images_per_class": {"min": int(counts.min()), "median": int(counts.median()), "max": int(counts.max())},
                "num_classes": NUM_CLASSES, "img_size": IMG_SIZE, "resize_mode_non_square": RESIZE_MODE,
                "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)}, "seed": SEED}
json.dump(prep_summary, open(PREP_OUT / "preprocessing_summary.json", "w"), indent=2)
prep_summary

### Shared evaluation functions
All four models are scored the same way on the same test images: accuracy, macro precision/recall/F1 (every Object ID weighted equally), and top-5 accuracy for classifiers. For detectors, the class of the most confident detection is the image's prediction (no detection = wrong).

In [ ]:
def image_level_metrics(y_true, y_pred, logits=None):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=np.arange(NUM_CLASSES), average="macro", zero_division=0)
    m = {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p, "macro_recall": r, "macro_f1": f1}
    if logits is not None: m["top5_accuracy"] = top_k_accuracy_score(y_true, logits, k=5, labels=np.arange(NUM_CLASSES))
    return m

def per_class_report(y_true, y_pred, path, title):
    rep = pd.DataFrame(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=[disp(c) for c in classes],
                                             output_dict=True, zero_division=0)).T
    rep.to_csv(path)
    print(f"\n=== {title} — 8 weakest Object IDs (test F1) ===")
    display(rep.iloc[:NUM_CLASSES].sort_values("f1-score").head(8).round(3))

def plot_confusion(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(cm, cmap="Blues"); plt.colorbar(im, fraction=0.046)
    ticks = [disp(c, 14) for c in classes]
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(ticks, rotation=90, fontsize=5); ax.set_yticklabels(ticks, fontsize=5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()
    off = cm.copy(); np.fill_diagonal(off, 0)
    print("Most confused pairs (true -> predicted : count)")
    for i, j in np.dstack(np.unravel_index(np.argsort(off, axis=None)[::-1][:10], off.shape))[0]:
        if off[i, j] > 0: print(f"  {disp(classes[i])} -> {disp(classes[j])} : {off[i, j]}")

---
# Part B — Classification: Custom CNN vs. EfficientNet-B0

- **Normalization:** ImageNet mean/std. **Training augmentation:** random resized crop, horizontal flip, ±15° rotation, color jitter. Validation/test: no augmentation.
- **Training:** AdamW, one-cycle learning-rate schedule, label smoothing 0.1, mixed precision, early stopping on validation macro-F1 (best checkpoint kept).
- Pretrained model: lower learning rate on the backbone, higher on the new 73-class head.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.img.tolist(), df.y.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

BATCH_SIZE = 64
def make_loader(df, tf, shuffle):
    return DataLoader(ImgDS(df, tf), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0)
cls_train_dl, cls_val_dl, cls_test_dl = make_loader(train_df, train_tf, True), make_loader(val_df, eval_tf, False), make_loader(test_df, eval_tf, False)

class CustomCNN(nn.Module):
    # 5 blocks of [conv-BN-ReLU x2 + maxpool] (32-64-128-256-256 filters), global average pooling, dropout 0.3, linear head
    def __init__(self, n):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                 nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), block(256, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n))
    def forward(self, x): return self.head(self.features(x))

def build_classifier(name):
    if name == "custom_cnn": return CustomCNN(NUM_CLASSES), "head"
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES); return m, "classifier"
    raise ValueError(name)

# name: (display name, max epochs, backbone LR, head LR)
CLS_MODELS = {"custom_cnn": ("Custom CNN", 40, 1e-3, 1e-3), "efficientnet_b0": ("EfficientNet-B0", 15, 2e-4, 2e-3)}
PATIENCE = 6

In [ ]:
@torch.no_grad()
def predict_cls(model, dl):
    model.eval(); L, Y = [], []
    for x, y in dl:
        with amp(): L.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
        Y.append(y)
    return torch.cat(L).numpy(), torch.cat(Y).numpy()

def train_classifier(name):
    disp_name, epochs, lr_bb, lr_head = CLS_MODELS[name]
    model, head_prefix = build_classifier(name); model.to(DEVICE)
    wpath, done = CLS_DIR / f"{name}_best.pt", CLS_DIR / f"{name}_done.json"
    if wpath.exists() and done.exists():                       # resume after a disconnect
        model.load_state_dict(torch.load(wpath, map_location=DEVICE)); print(f"{disp_name}: loaded saved model.")
        return model, pd.read_csv(CLS_DIR / f"{name}_history.csv"), json.load(open(done))["train_time_min"]
    seed_everything()
    head = [p for n_, p in model.named_parameters() if n_.startswith(head_prefix)]
    body = [p for n_, p in model.named_parameters() if not n_.startswith(head_prefix)]
    opt = torch.optim.AdamW([{"params": body, "lr": lr_bb}, {"params": head, "lr": lr_head}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[lr_bb, lr_head], epochs=epochs, steps_per_epoch=len(cls_train_dl), pct_start=0.15)
    crit, scaler = nn.CrossEntropyLoss(label_smoothing=0.1), torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
    hist, best_f1, best_state, bad, t0 = [], -1, None, 0, time.time()
    print(f"Training {disp_name} on {DEVICE} for up to {epochs} epochs ...", flush=True)
    for ep in range(1, epochs + 1):
        model.train(); tot = correct = n = 0; te = time.time()
        for x, y in tqdm(cls_train_dl, desc=f"{disp_name} ep{ep}", leave=False):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp(): out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
        vl, vy = predict_cls(model, cls_val_dl); vm = image_level_metrics(vy, vl.argmax(1))
        vloss = nn.functional.cross_entropy(torch.tensor(vl), torch.tensor(vy)).item()
        hist.append({"epoch": ep, "train_loss": tot / n, "train_acc": correct / n, "val_loss": vloss, "val_acc": vm["accuracy"], "val_macro_f1": vm["macro_f1"]})
        print(f"{disp_name} ep{ep:02d} | train loss {tot/n:.3f} acc {correct/n:.3f} | val loss {vloss:.3f} acc {vm['accuracy']:.3f} F1 {vm['macro_f1']:.3f} | {time.time()-te:.0f}s", flush=True)
        if vm["macro_f1"] > best_f1:
            best_f1, bad, best_state = vm["macro_f1"], 0, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: print("Early stopping."); break
    model.load_state_dict(best_state); torch.save(best_state, wpath)
    pd.DataFrame(hist).to_csv(CLS_DIR / f"{name}_history.csv", index=False)
    ttime = (time.time() - t0) / 60; json.dump({"train_time_min": ttime}, open(done, "w"))
    return model, pd.DataFrame(hist), ttime

def ms_per_image(model, n_batches=10):
    model.eval(); x = torch.randn(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.no_grad(), amp():
        for _ in range(3): model(x)
        sync(); t = time.time()
        for _ in range(n_batches): model(x)
        sync()
    return (time.time() - t) / (n_batches * BATCH_SIZE) * 1000

In [ ]:
cls_results, cls_hist, cls_test_pred = [], {}, {}
for name, (disp_name, *_ ) in CLS_MODELS.items():
    model, hist, ttime = train_classifier(name)
    vl, vy = predict_cls(model, cls_val_dl); tl, ty = predict_cls(model, cls_test_dl)
    vm, tm = image_level_metrics(vy, vl.argmax(1), vl), image_level_metrics(ty, tl.argmax(1), tl)
    cls_results.append({"model": disp_name, "task": "classification", "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
                        "epochs_run": len(hist), "train_time_min": ttime, "ms_per_img": ms_per_image(model),
                        **{f"val_{k}": v for k, v in vm.items()}, **{f"test_{k}": v for k, v in tm.items()}})
    cls_hist[name], cls_test_pred[name] = hist, tl
    del model; free_mem()
cls_res = pd.DataFrame(cls_results); cls_res.to_csv(CLS_DIR / "classification_results.csv", index=False)
cls_res.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for name, h in cls_hist.items():
    dn = CLS_MODELS[name][0]
    axes[0].plot(h.epoch, h.train_loss, "--", label=f"{dn} train"); axes[0].plot(h.epoch, h.val_loss, label=f"{dn} val")
    axes[1].plot(h.epoch, h.train_acc, "--", label=f"{dn} train"); axes[1].plot(h.epoch, h.val_acc, label=f"{dn} val")
    axes[2].plot(h.epoch, h.val_macro_f1, label=dn)
for ax, t in zip(axes, ["Loss", "Accuracy", "Validation macro-F1"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(CLS_DIR / "classification_training_curves.png", dpi=150); plt.show()

for name, tl in cls_test_pred.items():
    dn = CLS_MODELS[name][0]; tp = tl.argmax(1)
    per_class_report(test_df.y.values, tp, CLS_DIR / f"{name}_per_class_report.csv", dn)
    plot_confusion(test_df.y.values, tp, f"Confusion matrix — {dn} (test set)", CLS_DIR / f"confusion_{name}.png")

---
# Part C — Object Detection: YOLO11s vs. Faster R-CNN

### Bounding boxes
The dataset has **one Object ID label per image but no bounding boxes**, which detectors need for training. Boxes are generated automatically with **OWLv2**, an open-vocabulary detector that locates an object from a text prompt (e.g. *"a photo of a Basketball"*, using the object names from the official list). Each image shows one object, so the highest-scoring box is kept. If OWLv2 is not confident (score < 0.10) or the box is implausibly small, the whole image is used as the box (*fallback*); fallbacks are counted and reported.

### Evaluation
- **Image-level accuracy / macro-F1** against the true Object ID labels — directly comparable with Part B.
- **mAP@0.5 and mAP@0.5:0.95** (COCO-style, identical evaluator for both detectors) — measured against the generated boxes, i.e. agreement with pseudo-labels rather than hand-annotated ground truth.

## C1. Generate bounding boxes with OWLv2 (cached to Drive, resumable)

In [ ]:
OWL_ID, BOX_CACHE = "google/owlv2-base-patch16-ensemble", DET_DIR / "pseudo_boxes.csv"
MIN_SCORE, MIN_AREA, OWL_BATCH = 0.10, 0.005, 8

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
done = pd.read_csv(BOX_CACHE) if BOX_CACHE.exists() else pd.DataFrame(columns=["rel_path"])
todo = all_df[~all_df.rel_path.isin(set(done.rel_path))]
print(f"Boxes cached: {len(done)} | to generate: {len(todo)}")

def prompt_for(label):
    name = OBJECT_NAMES.get(label)
    return [f"a photo of a {name}" if name else "an object", "an object"]

if len(todo):
    from transformers import Owlv2Processor, Owlv2ForObjectDetection
    proc = Owlv2Processor.from_pretrained(OWL_ID)
    owl = Owlv2ForObjectDetection.from_pretrained(OWL_ID).to(DEVICE).eval()
    S, rows = IMG_SIZE, []
    for bi, i in enumerate(tqdm(range(0, len(todo), OWL_BATCH), desc="OWLv2 boxes")):
        chunk = todo.iloc[i:i + OWL_BATCH]
        imgs = [Image.open(p).convert("RGB") for p in chunk.img]
        inputs = proc(text=[prompt_for(l) for l in chunk.label], images=imgs, return_tensors="pt").to(DEVICE)
        with torch.no_grad(), amp(): out = owl(**inputs)
        scores = out.logits.float().sigmoid().max(-1).values
        best = scores.argmax(1)
        for b, r in enumerate(chunk.itertuples()):
            s = scores[b, best[b]].item()
            cx, cy, w, h = out.pred_boxes[b, best[b]].float().tolist()       # normalized; images are square
            x1, y1, x2, y2 = max(0., (cx - w/2) * S), max(0., (cy - h/2) * S), min(S, (cx + w/2) * S), min(S, (cy + h/2) * S)
            fallback = s < MIN_SCORE or (x2 - x1) * (y2 - y1) / S**2 < MIN_AREA or x2 - x1 < 2 or y2 - y1 < 2
            if fallback: x1, y1, x2, y2 = 0., 0., float(S), float(S)
            rows.append({"rel_path": r.rel_path, "x1": x1, "y1": y1, "x2": x2, "y2": y2, "owl_score": s, "fallback": fallback})
        if bi % 50 == 0 or i + OWL_BATCH >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); rows = []; done.to_csv(BOX_CACHE, index=False)
    del owl; free_mem()

boxes = pd.read_csv(BOX_CACHE).drop_duplicates("rel_path")
det_train, det_val, det_test = [d.merge(boxes, on="rel_path", how="left") for d in (train_df, val_df, test_df)]
assert not pd.concat([det_train, det_val, det_test]).x1.isna().any(), "some images have no box"
allb = pd.concat([det_train, det_val, det_test]); allb["fallback"] = allb.fallback.astype(str).str.lower() == "true"
for d in (det_train, det_val, det_test): d["fallback"] = d.fallback.astype(str).str.lower() == "true"
fb_rate = allb.fallback.mean()
print(f"Full-image fallback boxes: {fb_rate:.1%} of images")
fb = allb.groupby("label").fallback.mean().sort_values(ascending=False)
print("Object IDs with the most fallbacks:"); print(fb.head(10).rename(index=disp).map("{:.0%}".format).to_string())
fb.to_csv(DET_DIR / "pseudo_box_fallback_by_class.csv")

s = det_train.sample(24, random_state=SEED)
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
for ax, r in zip(axes.flat, s.itertuples()):
    ax.imshow(Image.open(r.img)); ax.axis("off")
    ax.add_patch(patches.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=2, edgecolor="orange" if r.fallback else "lime"))
    ax.set_title(f"{disp(r.label, 14)}\nscore {r.owl_score:.2f}", fontsize=7)
fig.suptitle("Generated boxes (green = OWLv2, orange = full-image fallback)"); plt.tight_layout()
plt.savefig(DET_DIR / "pseudo_boxes_examples.png", dpi=120); plt.show()

## C2. YOLO11s

In [ ]:
import yaml
for split, d in [("train", det_train), ("val", det_val), ("test", det_test)]:
    (YOLO_DIR / split / "images").mkdir(parents=True, exist_ok=True); (YOLO_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    for r in d.itertuples():
        stem = Path(r.rel_path).stem
        dst = YOLO_DIR / split / "images" / f"{stem}.jpg"
        if not dst.exists(): shutil.copy(r.img, dst)
        cx, cy, w, h = (r.x1 + r.x2) / 2 / IMG_SIZE, (r.y1 + r.y2) / 2 / IMG_SIZE, (r.x2 - r.x1) / IMG_SIZE, (r.y2 - r.y1) / IMG_SIZE
        (YOLO_DIR / split / "labels" / f"{stem}.txt").write_text(f"{int(r.y)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
DATA_YAML = YOLO_DIR / "data.yaml"
yaml.safe_dump({"path": str(YOLO_DIR.resolve()), "train": "train/images", "val": "val/images", "test": "test/images",
                "names": {i: c for i, c in enumerate(classes)}}, open(DATA_YAML, "w"), sort_keys=False)
print("YOLO dataset:", {s_: len(list((YOLO_DIR / s_ / "labels").glob("*.txt"))) for s_ in ["train", "val", "test"]})

In [ ]:
from ultralytics import YOLO
YOLO_EPOCHS = 60
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else DEVICE.type
YOLO_DONE, YOLO_BEST = DET_DIR / "yolo11s_done.json", DET_DIR / "yolo11s" / "weights" / "best.pt"
if YOLO_DONE.exists() and YOLO_BEST.exists():
    print("YOLO11s: loaded saved model."); yolo_train_min = json.load(open(YOLO_DONE))["train_time_min"]
else:
    t0 = time.time()
    YOLO("yolo11s.pt").train(data=str(DATA_YAML), imgsz=IMG_SIZE, epochs=YOLO_EPOCHS, batch=64, patience=15, seed=SEED, deterministic=True,
                             workers=NUM_WORKERS, device=YOLO_DEVICE, project=str(DET_DIR), name="yolo11s", exist_ok=True, plots=True, verbose=False)
    yolo_train_min = (time.time() - t0) / 60; json.dump({"train_time_min": yolo_train_min}, open(YOLO_DONE, "w"))
yolo_best = YOLO(str(YOLO_BEST))

## C3. Faster R-CNN (ResNet50-FPN v2, COCO-pretrained)

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
FRCNN_SIZE, FRCNN_EPOCHS, FRCNN_BS, FRCNN_LR = 320, 12, 8, 0.01

class DetDS(Dataset):
    def __init__(self, df, train):
        self.df, self.train, self.jit = df.reset_index(drop=True), train, transforms.ColorJitter(0.2, 0.2, 0.2, 0.02)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; img = Image.open(r.img).convert("RGB")
        box = torch.tensor([[r.x1, r.y1, r.x2, r.y2]], dtype=torch.float32)
        if self.train:
            img = self.jit(img)
            if random.random() < 0.5: img = TF.hflip(img); box[:, [0, 2]] = IMG_SIZE - box[:, [2, 0]]
        return TF.to_tensor(img), {"boxes": box, "labels": torch.tensor([int(r.y) + 1])}     # label 0 = background

collate = lambda b: tuple(zip(*b))
def det_loader(df, train, bs=FRCNN_BS):
    return DataLoader(DetDS(df, train), batch_size=bs, shuffle=train, num_workers=NUM_WORKERS, collate_fn=collate)

def build_frcnn():
    m = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT, min_size=FRCNN_SIZE, max_size=FRCNN_SIZE)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES + 1)
    return m

@torch.no_grad()
def frcnn_predict(model, df, bs=16):
    model.eval(); model.roi_heads.score_thresh, model.roi_heads.detections_per_img = 0.001, 100
    preds = []
    for imgs, _ in det_loader(df, False, bs):
        with amp(): out = model([im.to(DEVICE) for im in imgs])
        preds += [{"boxes": o["boxes"].float().cpu().numpy(), "scores": o["scores"].float().cpu().numpy(),
                   "labels": o["labels"].cpu().numpy() - 1} for o in out]
    return preds

def top_label(preds):
    return np.array([int(p["labels"][p["scores"].argmax()]) if len(p["scores"]) else -1 for p in preds])

In [ ]:
FRCNN_W, FRCNN_DONE = DET_DIR / "faster_rcnn_best.pt", DET_DIR / "faster_rcnn_done.json"
frcnn = build_frcnn().to(DEVICE)
if FRCNN_W.exists() and FRCNN_DONE.exists():
    frcnn.load_state_dict(torch.load(FRCNN_W, map_location=DEVICE)); print("Faster R-CNN: loaded saved model.")
    frcnn_train_min = json.load(open(FRCNN_DONE))["train_time_min"]; frcnn_hist = pd.read_csv(DET_DIR / "faster_rcnn_history.csv")
else:
    seed_everything()
    opt = torch.optim.SGD([p for p in frcnn.parameters() if p.requires_grad], lr=FRCNN_LR, momentum=0.9, weight_decay=1e-4)
    det_train_dl = det_loader(det_train, True)
    total, warm = FRCNN_EPOCHS * len(det_train_dl), min(500, len(det_train_dl))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s_: (s_ + 1) / warm if s_ < warm else 0.5 * (1 + np.cos(np.pi * (s_ - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
    hist, best_f1, best_state, t0 = [], -1, None, time.time()
    print(f"Training Faster R-CNN on {DEVICE} for {FRCNN_EPOCHS} epochs ...", flush=True)
    for ep in range(1, FRCNN_EPOCHS + 1):
        frcnn.train(); run = n = 0; te = time.time()
        for imgs, tg in tqdm(det_train_dl, desc=f"Faster R-CNN ep{ep}", leave=False):
            imgs = [im.to(DEVICE) for im in imgs]; tg = [{k: v.to(DEVICE) for k, v in t.items()} for t in tg]
            with amp(): loss = sum(frcnn(imgs, tg).values())
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item(); n += 1
        vm = image_level_metrics(det_val.y.values, top_label(frcnn_predict(frcnn, det_val)))
        hist.append({"epoch": ep, "train_loss": run / n, "val_acc": vm["accuracy"], "val_macro_f1": vm["macro_f1"]})
        print(f"Faster R-CNN ep{ep:02d} | train loss {run/n:.3f} | val acc {vm['accuracy']:.3f} F1 {vm['macro_f1']:.3f} | {time.time()-te:.0f}s", flush=True)
        if vm["macro_f1"] > best_f1:
            best_f1, best_state = vm["macro_f1"], {k: v.detach().cpu().clone() for k, v in frcnn.state_dict().items()}
    frcnn.load_state_dict(best_state); torch.save(best_state, FRCNN_W)
    frcnn_hist = pd.DataFrame(hist); frcnn_hist.to_csv(DET_DIR / "faster_rcnn_history.csv", index=False)
    frcnn_train_min = (time.time() - t0) / 60; json.dump({"train_time_min": frcnn_train_min}, open(FRCNN_DONE, "w"))

## C4. Evaluate both detectors on the test set

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

def yolo_predict(model, df):
    preds = []
    for r in model.predict(source=df.img.tolist(), imgsz=IMG_SIZE, conf=0.001, max_det=100, stream=True, verbose=False, device=YOLO_DEVICE):
        b = r.boxes
        preds.append({"boxes": b.xyxy.cpu().numpy(), "scores": b.conf.cpu().numpy(), "labels": b.cls.cpu().numpy().astype(int)})
    return preds

def box_iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def evaluate_detector(name, model, preds, ms_img, train_min, df=det_test):
    mapm = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    mapm.update([{"boxes": torch.as_tensor(p["boxes"], dtype=torch.float32).reshape(-1, 4), "scores": torch.as_tensor(p["scores"], dtype=torch.float32),
                  "labels": torch.as_tensor(p["labels"], dtype=torch.int64)} for p in preds],
                [{"boxes": torch.tensor([[r.x1, r.y1, r.x2, r.y2]], dtype=torch.float32), "labels": torch.tensor([int(r.y)])} for r in df.itertuples()])
    mp = mapm.compute(); y_pred = top_label(preds)
    ious = [box_iou(p["boxes"][p["scores"].argmax()], (r.x1, r.y1, r.x2, r.y2)) if len(p["scores"]) else 0. for p, r in zip(preds, df.itertuples())]
    m = image_level_metrics(df.y.values, y_pred)
    return {"model": name, "task": "detection", "params_M": sum(p.numel() for p in model.parameters()) / 1e6,
            "train_time_min": train_min, "ms_per_img": ms_img, **{f"test_{k}": v for k, v in m.items()},
            "test_mAP50": float(mp["map_50"]), "test_mAP50_95": float(mp["map"]), "test_mean_IoU_top_box": float(np.mean(ious)),
            "test_no_detection_rate": float((y_pred == -1).mean())}, y_pred

t = time.time(); yolo_preds = yolo_predict(yolo_best, det_test); yolo_ms = (time.time() - t) / len(det_test) * 1000
t = time.time(); frcnn_preds = frcnn_predict(frcnn, det_test); frcnn_ms = (time.time() - t) / len(det_test) * 1000
r1, yolo_y = evaluate_detector("YOLO11s", yolo_best.model, yolo_preds, yolo_ms, yolo_train_min)
r2, frcnn_y = evaluate_detector("Faster R-CNN", frcnn, frcnn_preds, frcnn_ms, frcnn_train_min)
det_res = pd.DataFrame([r1, r2]); det_res.to_csv(DET_DIR / "detection_results.csv", index=False)
det_res.round(4)

In [ ]:
for tag, name, yp in [("yolo11s", "YOLO11s", yolo_y), ("faster_rcnn", "Faster R-CNN", frcnn_y)]:
    per_class_report(det_test.y.values, yp, DET_DIR / f"{tag}_per_class_report.csv", name)
    plot_confusion(det_test.y.values, yp, f"Confusion matrix — {name} (test set, top detection)", DET_DIR / f"confusion_{tag}.png")
    if (yp < 0).any(): print(f"  note: {(yp < 0).sum()} test images had no detection (counted as errors)")

# Example predictions: generated box (green) vs. top predicted box (blue = correct class, red = wrong class)
idx = det_test.sample(8, random_state=SEED).index
fig, axes = plt.subplots(2, 8, figsize=(18, 5.5))
for row, (name, preds) in enumerate([("YOLO11s", yolo_preds), ("Faster R-CNN", frcnn_preds)]):
    for ax, i in zip(axes[row], idx):
        r, p = det_test.loc[i], preds[i]
        ax.imshow(Image.open(r.img)); ax.axis("off")
        ax.add_patch(patches.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=1.5, edgecolor="lime"))
        if len(p["scores"]):
            k = p["scores"].argmax(); b = p["boxes"][k]; correct = int(p["labels"][k]) == r.y
            ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=2, edgecolor="dodgerblue" if correct else "red"))
            ax.set_title(f"{name}\n{disp(classes[int(p['labels'][k])], 12)} {p['scores'][k]:.2f}", fontsize=6.5)
plt.tight_layout(); plt.savefig(DET_DIR / "detection_examples.png", dpi=130); plt.show()

---
# Part D — Final comparison of all four models
Same 1,098-image test set, same metrics, scored against the true Object ID labels.

In [ ]:
final = pd.concat([cls_res, det_res], ignore_index=True)
cols = ["model", "task", "params_M", "train_time_min", "ms_per_img", "test_accuracy", "test_macro_precision", "test_macro_recall",
        "test_macro_f1", "test_top5_accuracy", "test_mAP50", "test_mAP50_95", "test_mean_IoU_top_box"]
final = final[[c for c in cols if c in final.columns]]
final.to_csv(OUTPUT_ROOT / "final_comparison.csv", index=False)
display(final.round(4))

fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(final)); w = 0.38
ax.bar(x - w/2, final.test_accuracy, w, label="Test accuracy"); ax.bar(x + w/2, final.test_macro_f1, w, label="Test macro-F1")
for i, v in enumerate(final.test_accuracy): ax.text(i - w/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n({t})" for m, t in zip(final.model, final.task)], fontsize=9)
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis="y", alpha=.3); ax.set_title("All four models — test-set performance")
plt.tight_layout(); plt.savefig(OUTPUT_ROOT / "final_comparison.png", dpi=150); plt.show()

In [ ]:
# Everything needed for the report, in one place. Copy this output into the chat to get the report written.
best_cls = cls_res.sort_values("val_macro_f1", ascending=False).iloc[0].model
best_overall = final.sort_values("test_macro_f1", ascending=False).iloc[0].model
summary = {"preprocessing": prep_summary,
           "pseudo_box_fallback_rate": round(float(fb_rate), 4),
           "best_classifier_by_validation": best_cls, "highest_test_macro_f1": best_overall,
           "results": final.round(4).to_dict(orient="records"),
           "classifier_epochs_run": {r.model: int(r.epochs_run) for r in cls_res.itertuples()}}
json.dump(summary, open(OUTPUT_ROOT / "report_summary.json", "w"), indent=2)
print(json.dumps(summary, indent=2))
print("\nAll outputs saved to:", OUTPUT_ROOT)